# Tema 9 · Laboratorio — Métricas, matriz de confusión y ROC

**Aprendizaje Profundo · CUNEF Universidad**

Evaluamos un clasificador **más allá del accuracy** sobre un dataset **desbalanceado**. Veremos por qué el accuracy engaña, cómo leer la matriz de confusión, precision/recall/F1, la curva ROC-AUC y el efecto de mover el umbral.

Este laboratorio usa **scikit-learn** (no hace falta GPU): el foco es la evaluación.

> Ejecuta las celdas en orden.

## 1 · Un dataset desbalanceado

Creamos un problema binario donde solo el **~10 %** de los casos son positivos (como una enfermedad rara o un fraude).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report,
                             roc_curve, roc_auc_score, precision_score, recall_score)

X, y = make_classification(n_samples=4000, n_features=12, n_informative=5,
                           weights=[0.90, 0.10], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)
print('Positivos en test:', y_test.sum(), 'de', len(y_test),
      f'({100*y_test.mean():.1f} %)')

## 2 · Entrenar y caer en la trampa del accuracy

Entrenamos una regresión logística y miramos el accuracy… y lo comparamos con un modelo tonto que dice **siempre 'negativo'**.

In [ ]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc_modelo = accuracy_score(y_test, y_pred)
acc_tonto = accuracy_score(y_test, np.zeros_like(y_test))  # predice siempre 0
print(f'Accuracy del modelo         : {acc_modelo:.3f}')
print(f'Accuracy de "siempre negativo": {acc_tonto:.3f}')
print('\n-> Un accuracy alto NO basta: el modelo tonto ya roza ese valor sin detectar nada.')

## 3 · La matriz de confusión

Abrimos la caja: TP, FP, FN, TN. Es de donde salen todas las demás métricas.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f'TP={tp}  FP={fp}  FN={fn}  TN={tn}')

ConfusionMatrixDisplay(cm, display_labels=['Neg', 'Pos']).plot(cmap='Blues', colorbar=False)
plt.title('Matriz de confusión (umbral por defecto 0.5)')
plt.show()

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Negativo', 'Positivo']))
print('Fíjate en la fila "Positivo": ahí es donde el modelo lo pasa peor.')

## 4 · La curva ROC y el AUC

El AUC resume la calidad del modelo a través de **todos** los umbrales, sin fijar ninguno. A diferencia del accuracy, no se deja engañar por el desbalanceo.

In [ ]:
y_scores = clf.predict_proba(X_test)[:, 1]  # probabilidad de la clase positiva
fpr, tpr, thresholds = roc_curve(y_test, y_scores)
auc = roc_auc_score(y_test, y_scores)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color='#ff5700', lw=2.5, label=f'ROC (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], '--', color='gray', label='azar (AUC = 0.5)')
plt.xlabel('FPR (falsas alarmas)'); plt.ylabel('TPR (recall)')
plt.title('Curva ROC'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 5 · Mover el umbral

El umbral por defecto es 0.5, pero podemos elegir otro sobre `predict_proba`. Bajarlo sube el recall (pillamos más positivos) a costa de la precision, y al revés.

In [ ]:
print(f'{"umbral":>8} {"precision":>10} {"recall":>8} {"positivos marcados":>20}')
for t in [0.2, 0.35, 0.5, 0.65, 0.8]:
    pred_t = (y_scores >= t).astype(int)
    p = precision_score(y_test, pred_t, zero_division=0)
    r = recall_score(y_test, pred_t, zero_division=0)
    print(f'{t:>8.2f} {p:>10.3f} {r:>8.3f} {pred_t.sum():>20}')

print('\n-> Umbral bajo = mucho recall, poca precision. Umbral alto = al revés.')

## 6 · Tus retos

1. **Elige un umbral.** Si esto fuera un cribado médico (un FN es gravísimo), ¿qué umbral de la tabla elegirías y por qué?
2. **Reponderar clases.** Reentrena con `LogisticRegression(class_weight='balanced')`. ¿Mejora el recall de la clase positiva? ¿A costa de qué?
3. **Otro modelo.** Prueba un `RandomForestClassifier`. ¿Sube el AUC? Compara su matriz de confusión con la de la regresión logística.

Cuando termines, vuelve a la [práctica interactiva](../../practica-t9.html) y comprueba que mover el umbral produce el mismo tira y afloja que has medido aquí.